# MARS · Camera-conditioned video person re-identification

Does predicting three genuinely later frame embeddings improve identity retrieval from four observed crops? The two heads share the state of the observed prefix. Only the auxiliary temporal head rolls predictions forward. The view head predicts a destination-camera appearance and its scalar error scale from the observed state.

**Kaggle setup.** Import this notebook and run all cells in order. The default smoke profile uses generated tracks on CPU or GPU. For benchmark experiments attach the release, choose a GPU, switch to research and run all. Results accumulate under /kaggle/working. Short tracks use masked padding.

Attach the MARS release with bbox_train and bbox_test (directories or individual ZIPs) alongside info/ containing the filename lists, track tables and query_IDX.mat. DATA_ROOT points to their common parent. [MARS release](https://zheng-lab-anu.github.io/Project/project_mars.html) · [official evaluator](https://github.com/liangzheng06/MARS-evaluation).

**Reading.** [GLTR](https://openaccess.thecvf.com/content_ICCV_2019/html/Li_Global-Local_Temporal_Representations_for_Video_Person_Re-Identification_ICCV_2019_paper.html) motivates temporal identity descriptors. [TCViT](https://doi.org/10.1609/aaai.v38i6.28424) provides video ReID context. [V-JEPA 2](https://arxiv.org/abs/2506.09985) motivates prediction in embedding space. The CNN and GRUs here are trained from scratch.

## 1. Configure the run

For a limited GPU session, run a single seed or split at a time. Each run has its own output directory. Use all three seeds for the final comparison.

In [1]:
# 1. Imports and run settings
import re
from itertools import combinations
import io
import json
import math
import os
import random
import time
import zipfile
from collections import defaultdict, deque
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageDraw
from scipy.io import loadmat
from torch.utils.data import DataLoader, Dataset, Sampler

DATASET = "MARS"
RUN_PROFILE = "smoke"  # "smoke" or "research"
DATA_ROOT = ""         # Blank enables discovery under /kaggle/input.
QUERY_MAT = ""         # LS-VID only, optional official query index file.
SEEDS = [42] if RUN_PROFILE == "smoke" else [42, 43, 44]
SPLITS = [0]
SPLIT_SEED = 1729
NUM_WORKERS = 0
assert RUN_PROFILE in ("smoke","research")
assert SEEDS and SPLITS and len(set(SEEDS))==len(SEEDS) and len(set(SPLITS))==len(SPLITS)
assert all(i in range(1) for i in SPLITS)
OUTPUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".") / "reid" / DATASET / RUN_PROFILE
OUTPUT.mkdir(parents=True,exist_ok=True)

@dataclass(frozen=True)
class Settings:
    height: int
    width: int
    embed_dim: int
    hidden_dim: int
    encoder_width: int
    identities_per_batch: int
    frame_epochs: int
    head_epochs: int
    max_steps: int | None
    train_id_limit: int | None
    test_id_limit: int | None
    train_tracks_per_camera: int | None
    extract_batch: int
    context: int = 4
    horizon: int = 3
    stride: int = 2
    view_loss_weight: float = 0.50
    forecast_loss_weight: float = 0.75
    prefix_occlusion_probability: float = 0.30
    variance_reference: float = 0.20

PROFILES = {
    "smoke": Settings(48, 24, 32, 48, 12, 4, 2, 3, 5, None, None, None, 16),
    "pilot": Settings(96, 48, 96, 128, 24, 8, 5, 8, 75, 104, 64, 3, 32),
    "research": Settings(128, 64, 128, 192, 24, 12, 18, 24, None, None, None, None, 32),
}
cfg = PROFILES[RUN_PROFILE]
device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        torch.zeros(1, device="cuda").add_(1)
        torch.cuda.synchronize()
    except RuntimeError as error:
        print(f"CUDA probe failed. The notebook will use CPU. Choose a compatible GPU such as a T4.\n{error}")
    else:
        device = torch.device("cuda")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.set_float32_matmul_precision("high" if device.type == "cuda" else "highest")
torch.backends.cudnn.benchmark = False

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

use_amp = device.type == "cuda"
print(DATASET, RUN_PROFILE, SEEDS, SPLITS, device)

MARS smoke [42] [0] cuda


## 2. Read ordered tracklets and identity splits

Attach the MARS release with bbox_train and bbox_test (directories or individual ZIPs) alongside info/ containing the filename lists, track tables and query_IDX.mat. DATA_ROOT points to their common parent. [MARS release](https://zheng-lab-anu.github.io/Project/project_mars.html) · [official evaluator](https://github.com/liangzheng06/MARS-evaluation).

In [2]:
@dataclass(frozen=True)
class Tracklet:
    split: str
    row: int
    pid: int
    cam: int
    frames: tuple

    @property
    def key(self):
        return f"{self.split}-{self.row:06d}"

def group_by_identity_and_camera(tracks):
    groups = defaultdict(lambda: defaultdict(list))
    for index, track in enumerate(tracks):
        if track.pid > 0:
            groups[track.pid][track.cam].append(index)
    return {pid: dict(by_cam) for pid, by_cam in groups.items()}

def choose_ids(pids, count, rng):
    pids = sorted(pids)
    if count is None or len(pids) <= count:
        return set(pids)
    return set(rng.choice(pids, count, replace=False).tolist())

def make_val_queries(tracks):
    groups = group_by_identity_and_camera(tracks)
    return [
        tracks[indices[0]]
        for by_camera in groups.values() if len(by_camera) > 1
        for indices in by_camera.values()
    ]

def limit_train_tracks(tracks, limit, rng):
    if limit is None:
        return tracks
    keep = []
    for by_camera in group_by_identity_and_camera(tracks).values():
        for indices in by_camera.values():
            selected = rng.choice(indices, min(limit, len(indices)), replace=False)
            keep.extend(selected.tolist())
    return [tracks[index] for index in sorted(keep)]

def synthetic_tracks(split, pids, seed):
    rng = np.random.default_rng(seed)
    tracks = []
    for pid in pids:
        for cam in range(3):
            for repeat in range(2):
                row = len(tracks)
                length = int(rng.integers(8, 23))
                tracks.append(Tracklet(split, row, pid, cam, tuple(range(length))))
    return tracks


def locate(filename):
    base = Path(DATA_ROOT or "/kaggle/input")
    hits = list(base.rglob(filename))
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {filename} under {base}, found {len(hits)}. Set DATA_ROOT to the extracted release.")
    return hits[0]

def images_by_number(directory):
    def order(path):
        return [int(tok) if tok.isdigit() else tok.lower() for tok in re.split(r'(\d+)', path.name)]
    images = sorted((p for p in directory.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'}), key=order)
    if not images:
        raise ValueError(f"No frames in {directory}")
    return tuple(images)

def split_training(tracks, split):
    rng = np.random.default_rng(SPLIT_SEED + split)
    groups = group_by_identity_and_camera(tracks)
    eligible = {pid for pid, cams in groups.items() if len(cams) >= 2}
    if len(eligible) < 4:
        raise ValueError("Need at least four multi-camera development identities")
    n = min(max(2, round(0.12 * len(eligible))), len(eligible)-2)
    held = choose_ids(eligible, n, rng)
    return [t for t in tracks if t.pid in eligible-held], [t for t in tracks if t.pid in held]

def validation_views(tracks):
    return make_val_queries(tracks), tracks
INFO_FILES = (
    "train_name.txt", "test_name.txt", "tracks_train_info.mat",
    "tracks_test_info.mat", "query_IDX.mat",
)

def split_input(root, split):
    folder = root / f"bbox_{split}"
    archive = root / f"bbox_{split}.zip"
    if folder.is_dir():
        return folder
    if archive.is_file():
        return archive
    raise FileNotFoundError(f"Missing {folder.name}/ or {archive.name} in {root}")

def is_mars_root(path):
    if not all((path / "info" / name).is_file() for name in INFO_FILES):
        return False
    return all(
        (path / f"bbox_{split}").is_dir() or (path / f"bbox_{split}.zip").is_file()
        for split in ("train", "test")
    )

def find_mars_root(explicit):
    if explicit:
        root = Path(explicit).expanduser()
        if is_mars_root(root):
            return root
        raise FileNotFoundError(f"{root} is missing one or more inputs in the setup diagram.")
    pending = deque([(Path("/kaggle/input"), 0)])
    matches = []
    while pending:
        candidate, depth = pending.popleft()
        if not candidate.is_dir():
            continue
        if is_mars_root(candidate):
            matches.append(candidate)
            continue
        if depth < 5:
            pending.extend(
                (child, depth + 1) for child in sorted(candidate.iterdir())
                if child.is_dir() and child.name not in {"bbox_train", "bbox_test", "info"}
            )
    if len(matches) == 1:
        return matches[0]
    detail = ", ".join(str(path) for path in matches) or "no matching folder"
    raise FileNotFoundError(f"MARS auto-discovery found {detail}. Check the layout and set DATA_ROOT.")

def read_mat_integer(path, key):
    arrays = loadmat(path, variable_names=[key])
    if key not in arrays:
        raise ValueError(f"{path.name} does not contain '{key}'.")
    data = arrays[key]
    if not np.isfinite(data).all() or not np.equal(data, np.rint(data)).all():
        raise ValueError(f"{path.name} contains non-integer or non-finite values.")
    return data.astype(np.int64)

def read_mars_tracks(info, split):
    name_file = info / f"{split}_name.txt"
    names = [line.strip() for line in name_file.read_text(encoding="utf-8-sig").splitlines()]
    rows = read_mat_integer(info / f"tracks_{split}_info.mat", f"track_{split}_info")
    if rows.ndim != 2 or rows.shape[1] != 4:
        raise ValueError(f"Expected four columns in the official {split} tracklet table.")
    tracks = []
    for row, (first, last, pid, cam) in enumerate(rows):
        if not 1 <= first <= last <= len(names) or not 1 <= cam <= 6:
            raise ValueError(f"Invalid MARS frame bounds or camera in {split} row {row + 1}.")
        filenames = tuple(names[first - 1:last])
        if not filenames[0] or not filenames[-1]:
            raise ValueError(f"Empty frame filename in {split} row {row + 1}.")
        tracks.append(Tracklet(split, row, int(pid), int(cam - 1), filenames))
    return tracks


def load_real(split):
    root = find_mars_root(DATA_ROOT)
    paths = {s: split_input(root,s) for s in ('train','test')}
    train,val = split_training(read_mars_tracks(root/'info','train'), split)
    test = read_mars_tracks(root/'info','test')
    rows = read_mat_integer(root/'info'/'query_IDX.mat','query_IDX').ravel()-1
    assert len(set(rows)) == len(rows) and rows.min() >= 0 and rows.max() < len(test)
    q = [test[i] for i in rows]
    assert all(t.pid > 0 for t in q)
    return train, *validation_views(val), q, [t for t in test if t.pid >= 0], paths, 6, "release query indices / all non-junk test tracks"

def load_split(split):
    if RUN_PROFILE == 'smoke':
        cams = 2 if DATASET == 'iLIDS-VID' else 3
        def toy(role,pids):
            rows = synthetic_tracks(role,pids,SPLIT_SEED)
            return [t for t in rows if t.cam<cams and (DATASET!='iLIDS-VID' or t.row%2==0)]
        train,val = split_training(toy('train',range(1,17)),split)
        test = toy('test',range(101,109))
        q = [t for t in test if t.cam==0 and t.row%2==0]
        g = test if DATASET=='MARS' else [t for t in test if t.cam!=0]
        vq,vg = validation_views(val)
        result = (train,vq,vg,q,g,{},cams,'artificial pipeline check')
    else:
        result = load_real(split)
    train,vq,vg,tq,tg,paths,ncam,protocol = result
    eligible = {pid for pid,cams in group_by_identity_and_camera(train).items() if len(cams)>=2}
    train = [t for t in train if t.pid in eligible]
    train_ids,val_ids,test_ids = [{t.pid for t in tracks if t.pid>0} for tracks in (train,vq+vg,tq+tg)]
    assert train_ids.isdisjoint(val_ids|test_ids) and val_ids.isdisjoint(test_ids)
    assert all((train,vq,vg,tq,tg))
    print(protocol, '| train / validation / test identities:',len(train_ids),len(val_ids),len(test_ids))
    return train,vq,vg,tq,tg,paths,ncam,protocol

## 3. Sample crops and simulate query occlusion

Four observed positions precede three targets. If stride two would discard available frames, sample adjacent frames. Training and test metrics only use valid positions. The rectangle covers roughly 48% of height and 55% of width on up to two observed crops.

In [3]:
# 3. Read only the sampled crops, preserve their order, and mask padding.
class FrameReader:
    def __init__(self, paths):
        self.paths = paths
        self._process = None
        self._archives = {}
        self._prefixes = {}

    def close(self):
        for handle in self._archives.values():
            handle.close()
        self._archives.clear()
        self._prefixes.clear()

    def _read_from_zip(self, path, relative):
        if os.getpid() != self._process:
            self.close()
            self._process = os.getpid()
        if path not in self._archives:
            handle = zipfile.ZipFile(path)
            self._archives[path] = handle
            sample = next(
                (entry.filename for entry in handle.filelist
                 if entry.filename.lower().endswith((".jpg", ".jpeg"))
                 and not entry.filename.startswith("__MACOSX/")),
                None,
            )
            if sample is None:
                raise ValueError(f"{path.name} contains no JPEG crops.")
            head, _, filename = sample.rpartition("/")
            prefix, _, pid_dir = head.rpartition("/")
            if pid_dir != filename[:4]:
                raise ValueError(f"Unexpected crop path in {path.name}: {sample}")
            self._prefixes[path] = prefix + "/" if prefix else ""
        handle = self._archives[path]
        return handle.read(self._prefixes[path] + relative)

    def read(self, track, frame_index):
        frame = track.frames[frame_index]
        if not self.paths:
            return self.synthetic_image(track, int(frame))
        folder = self.paths[track.split]
        relative = f"{frame[:4]}/{frame}"
        try:
            if folder.is_dir():
                with Image.open(folder / relative) as image:
                    return image.convert("RGB")
            with Image.open(io.BytesIO(self._read_from_zip(folder, relative))) as image:
                return image.convert("RGB")
        except (KeyError, OSError, zipfile.BadZipFile) as error:
            raise RuntimeError(f"Could not read {relative} from {folder} for {track.key}.") from error

    @staticmethod
    def synthetic_image(track, frame):
        rng = np.random.default_rng(track.pid)
        shirt = tuple(rng.integers(55, 220, 3).tolist())
        trousers = tuple(rng.integers(20, 155, 3).tolist())
        stripe = tuple(rng.integers(120, 248, 3).tolist())
        background = [(25, 38, 52), (58, 42, 32), (31, 53, 40)][track.cam]
        image = Image.new("RGB", (40, 80), background)
        draw = ImageDraw.Draw(image)
        wave = math.sin((frame + track.row * 0.27) * (0.22 + 0.03 * (track.pid % 5)))
        x = 20 + round(3 * wave) + track.cam - 1
        y = 9 + round(2 * wave)
        draw.ellipse((x - 5, y, x + 5, y + 10), fill=(188, 149, 106))
        draw.rounded_rectangle((x - 9, y + 11, x + 9, y + 37), 3, fill=shirt)
        draw.rectangle((x - 9, y + 16 + track.pid % 6, x + 9, y + 19 + track.pid % 6), fill=stripe)
        leg = round(4 * wave)
        draw.line((x - 4, y + 36, x - 6 - leg, y + 64), fill=trousers, width=7)
        draw.line((x + 4, y + 36, x + 6 + leg, y + 64), fill=trousers, width=7)
        return image

def sampled_indices(length, count, stride, fraction):
    if length < 1 + (count - 1) * stride:
        stride = 1
    last_start = max(0, length - (1 + (count - 1) * stride))
    start = random.randint(0, last_start) if fraction is None else round(fraction * last_start)
    indices = start + stride * np.arange(count)
    return np.minimum(indices, length - 1), indices < length

def occlude_prefix(images, valid, seed):
    """Hide a contiguous body region in up to half the valid context frames."""
    rng = np.random.default_rng(seed)
    copies = images.clone()
    candidates = np.flatnonzero(valid[:cfg.context])
    if len(candidates) == 0:
        return copies
    count = min(len(candidates), max(1, cfg.context // 2))
    chosen = rng.choice(candidates, count, replace=False)
    height, width = copies.shape[-2:]
    h, w = max(1, round(0.48 * height)), max(1, round(0.55 * width))
    top = int(rng.integers(height // 5, height - h + 1))
    left = int(rng.integers(0, width - w + 1))
    copies[chosen, :, top:top + h, left:left + w] = 0.5
    return copies

class ClipDataset(Dataset):
    def __init__(self, tracks, paths, count, fraction, train=False, with_occlusion=False):
        self.tracks = tracks
        self.count = count
        self.fraction = fraction
        self.train = train
        self.with_occlusion = with_occlusion
        self.reader = FrameReader(paths)

    def __len__(self):
        return len(self.tracks)

    def __getitem__(self, index):
        track = self.tracks[index]
        frame_ids, mask = sampled_indices(len(track.frames), self.count, cfg.stride, self.fraction)
        flip = self.train and random.random() < 0.5
        gain = random.uniform(0.85, 1.15) if self.train else 1.0
        frames = {}
        for frame_id in np.unique(frame_ids):
            image = self.reader.read(track, int(frame_id)).resize(
                (cfg.width, cfg.height), resample=Image.Resampling.BILINEAR
            )
            if flip:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            frames[frame_id] = np.asarray(image, dtype=np.uint8)
        images = torch.from_numpy(np.stack([frames[frame_id] for frame_id in frame_ids]))
        images = images.permute(0, 3, 1, 2).float().div_(255.0).mul_(gain).clamp_(0.0, 1.0)
        record = {"images": images, "mask": torch.from_numpy(mask), "index": index}
        if self.with_occlusion:
            corruption_seed = SPLIT_SEED + 97 * track.row + (1_000_003 if track.split == "test" else 0)
            record["occluded"] = occlude_prefix(images, mask, corruption_seed)
        return record

def make_loader(dataset, *, batch_size=None, batch_sampler=None, seed=SPLIT_SEED):
    options = {
        "num_workers": NUM_WORKERS,
        "pin_memory": device.type == "cuda",
        "generator": torch.Generator().manual_seed(seed),
    }
    if batch_sampler is not None:
        return DataLoader(dataset, batch_sampler=batch_sampler, **options)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, **options)

n_frames = cfg.context + cfg.horizon

## 4. Build the shared appearance encoder

Select two cameras and at least two identities seen in both. All examples have a same-ID cross-camera positive and destination-camera negatives. Unit-normalized embeddings feed a scaled ID classifier and a cosine triplet loss.

In [4]:
class CameraPairBatchSampler(Sampler):
    def __init__(self, pids, cams, seed, keep=None):
        pids = np.asarray(pids)
        cams = np.asarray(cams)
        keep = np.ones(len(pids), dtype=bool) if keep is None else np.asarray(keep, dtype=bool)
        self.groups = defaultdict(lambda: defaultdict(list))
        for index in np.flatnonzero(keep):
            self.groups[int(pids[index])][int(cams[index])].append(int(index))

        self.by_pair = defaultdict(list)
        for pid, cameras in self.groups.items():
            for pair in combinations(sorted(cameras), 2):
                self.by_pair[pair].append(pid)
        self.pairs = sorted(pair for pair, ids in self.by_pair.items() if len(ids) >= 2)
        if not self.pairs:
            raise ValueError("No camera pair has usable tracklets for two or more identities.")
        counts = np.array([len(self.by_pair[pair]) for pair in self.pairs], dtype=np.float64)
        self.probabilities = counts / counts.sum()
        self.steps = math.ceil(keep.sum() / (2 * cfg.identities_per_batch))
        if cfg.max_steps is not None:
            self.steps = min(self.steps, cfg.max_steps)
        self.seed = seed
        self.epoch = 0

    def __len__(self):
        return self.steps

    def __iter__(self):
        rng = np.random.default_rng(self.seed + 1009 * self.epoch)
        for _ in range(self.steps):
            pair = self.pairs[rng.choice(len(self.pairs), p=self.probabilities)]
            choices = self.by_pair[pair]
            pids = rng.choice(choices, min(cfg.identities_per_batch, len(choices)), replace=False)
            yield [
                int(rng.choice(self.groups[int(pid)][cam]))
                for pid in pids for cam in pair
            ]


def paired_index(batch_size, tensor_device):
    if batch_size % 2:
        raise ValueError("Every training item must have one paired item in its batch.")
    return torch.arange(batch_size, device=tensor_device) ^ 1

def masked_anchor(features, mask):
    summed = (features * mask.unsqueeze(-1)).sum(dim=1)
    return F.normalize(summed, dim=-1)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
            nn.GroupNorm(4, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.GroupNorm(4, out_channels),
        )
        self.skip = nn.Identity() if (stride == 1 and in_channels == out_channels) else nn.Conv2d(
            in_channels, out_channels, 1, stride, bias=False
        )

    def forward(self, images):
        return F.silu(self.body(images) + self.skip(images))

class FrameEncoder(nn.Module):
    def __init__(self, width, embed_dim, num_labels):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, width, 5, 2, 2, bias=False),
            nn.GroupNorm(4, width),
            nn.SiLU(),
            ResidualBlock(width, width, 1),
            ResidualBlock(width, 2 * width, 2),
            ResidualBlock(2 * width, 3 * width, 2),
            ResidualBlock(3 * width, 4 * width, 2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(4 * width, embed_dim),
            nn.LayerNorm(embed_dim),
        )
        self.classifier = nn.Linear(embed_dim, num_labels, bias=False)

    def forward(self, clips):
        batch, steps = clips.shape[:2]
        images = clips.flatten(0, 1).mul(2.0).sub(1.0)
        embedding = F.normalize(self.features(images).float(), dim=-1)
        return embedding.reshape(batch, steps, -1)

def appearance_objective(encoder, clips, mask, labels):
    features = encoder(clips)
    anchor = masked_anchor(features, mask)
    logits = 12.0 * F.linear(anchor, F.normalize(encoder.classifier.weight.float(), dim=-1))
    classification = F.cross_entropy(logits, labels)
    distances = 1.0 - (anchor @ anchor.T)
    paired = paired_index(len(anchor), anchor.device)
    positive_distance = distances[torch.arange(len(anchor), device=anchor.device), paired]
    negatives = labels[:, None] != labels[None, :]
    negative_distance = distances.masked_fill(~negatives, float("inf")).amin(dim=1)
    triplet = F.relu(positive_distance - negative_distance + 0.20).mean()
    return classification + 0.60 * triplet, classification, triplet

## 5. Train the appearance encoder

Within a seed and split, train one encoder and freeze it for every retrieval method.

In [5]:
def atomic_checkpoint(data, filename):
    path = Path(filename)
    temporary = path.with_suffix(".tmp")
    torch.save(data, temporary)
    temporary.replace(path)

def train_encoder(train_tracks, paths, seed, out_dir):
    class_to_index = {pid:i for i,pid in enumerate(sorted({t.pid for t in train_tracks}))}
    frame_model = FrameEncoder(cfg.encoder_width, cfg.embed_dim, len(class_to_index)).to(device)
    frame_dataset = ClipDataset(train_tracks, paths, cfg.context, fraction=None, train=True)
    frame_sampler = CameraPairBatchSampler([t.pid for t in train_tracks], [t.cam for t in train_tracks], seed + 1200)
    frame_loader = make_loader(frame_dataset, batch_sampler=frame_sampler, seed=seed + 3)
    train_class_indices = torch.tensor([class_to_index[track.pid] for track in train_tracks])

    frame_optimizer = torch.optim.AdamW(frame_model.parameters(), lr=8e-4, weight_decay=1e-4)
    frame_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        frame_optimizer, T_max=cfg.frame_epochs, eta_min=8e-5
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    frame_history = []
    for epoch in range(cfg.frame_epochs):
        frame_model.train()
        frame_sampler.epoch = epoch
        total = np.zeros(3, dtype=np.float64)
        for batch in frame_loader:
            clips = batch["images"].to(device, non_blocking=use_amp)
            mask = batch["mask"].to(device, non_blocking=use_amp)
            targets = train_class_indices[batch["index"]].to(device)
            frame_optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
                loss, classification, triplet = appearance_objective(frame_model, clips, mask, targets)
            if not bool(torch.isfinite(loss)):
                raise FloatingPointError("Non-finite appearance loss.")
            scaler.scale(loss).backward()
            scaler.unscale_(frame_optimizer)
            nn.utils.clip_grad_norm_(frame_model.parameters(), max_norm=5.0)
            scaler.step(frame_optimizer)
            scaler.update()
            total += torch.stack((loss, classification, triplet)).detach().float().cpu().numpy()
        frame_scheduler.step()
        averages = total / len(frame_loader)
        frame_history.append((epoch + 1, *averages.tolist()))
        print(f"Encoder epoch {epoch + 1:02d}/{cfg.frame_epochs:02d} | loss {averages[0]:.4f}")
    frame_dataset.reader.close()
    frame_model.eval().requires_grad_(False)

    atomic_checkpoint({
        "model_state": frame_model.state_dict(),
        "settings": asdict(cfg),
        "class_to_index": class_to_index,
    }, out_dir / "frame_encoder.pt")
    pd.DataFrame(
        frame_history, columns=["epoch", "total_loss", "classification_loss", "triplet_loss"]
    ).to_csv(out_dir / "appearance_history.csv", index=False)

    return frame_model

## 6. Cache clean and corrupted embeddings

Use two windows per training tracklet and one centered window for evaluation. The test suffix is used only by the forecast diagnostic.

In [6]:
# 6. Cache frozen clean latents and occluded context latents.
@dataclass
class Features:
    clean: torch.Tensor
    occluded: torch.Tensor
    valid: torch.Tensor
    cameras: torch.Tensor
    pids: torch.Tensor
    keys: tuple

    def take(self, indices):
        indices = torch.as_tensor(indices, dtype=torch.long)
        return Features(
            self.clean[indices], self.occluded[indices], self.valid[indices],
            self.cameras[indices], self.pids[indices],
            tuple(self.keys[i] for i in indices.tolist()),
        )

@torch.no_grad()
def cache_latents(frame_model, paths, tracks, fraction, label, report=True):
    dataset = ClipDataset(tracks, paths, n_frames, fraction, with_occlusion=True)
    loader = make_loader(dataset, batch_size=cfg.extract_batch)
    clean = torch.empty((len(tracks), n_frames, cfg.embed_dim), dtype=torch.float32)
    occluded = torch.empty((len(tracks), cfg.context, cfg.embed_dim), dtype=torch.float32)
    valid = torch.empty((len(tracks), n_frames), dtype=torch.bool)
    start = time.monotonic()
    for batch in loader:
        index = batch["index"]
        images = batch["images"].to(device, non_blocking=use_amp)
        corrupted = batch["occluded"][:, :cfg.context].to(device, non_blocking=use_amp)
        with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
            clean[index] = frame_model(images).float().cpu()
            occluded[index] = frame_model(corrupted).float().cpu()
        valid[index] = batch["mask"]
    dataset.reader.close()
    if not bool(torch.isfinite(clean).all() & torch.isfinite(occluded).all()):
        raise FloatingPointError(f"Non-finite features in {label}.")
    if report:
        future_fraction = valid[:, cfg.context:].any(1).float().mean().item()
        print(f"{label}: {len(tracks):,} clips | with future frames {future_fraction:.1%} "
              f"| {time.monotonic() - start:.1f}s")
    return Features(
        clean, occluded, valid,
        torch.tensor([track.cam for track in tracks], dtype=torch.long),
        torch.tensor([track.pid for track in tracks], dtype=torch.long),
        tuple(track.key for track in tracks),
    )

def cache_eval(encoder, paths, queries, gallery, label):
    g = cache_latents(encoder,paths,gallery,0.5,label+" gallery")
    lookup = {key:i for i,key in enumerate(g.keys)}
    if all(t.key in lookup for t in queries):
        q = g.take([lookup[t.key] for t in queries])
    else:
        q = cache_latents(encoder,paths,queries,0.5,label+" queries")
    return q,g

def cache_train(encoder,paths,train_tracks):
    cache_a = cache_latents(encoder, paths, train_tracks, 0.15, "Train windows A")
    cache_b = cache_latents(encoder, paths, train_tracks, 0.75, "Train windows B")
    train_feat = Features(
        torch.cat((cache_a.clean, cache_b.clean)),
        torch.cat((cache_a.occluded, cache_b.occluded)),
        torch.cat((cache_a.valid, cache_b.valid)),
        torch.cat((cache_a.cameras, cache_b.cameras)),
        torch.cat((cache_a.pids, cache_b.pids)),
        cache_a.keys + cache_b.keys,
    )
    del cache_a, cache_b
    if RUN_PROFILE != "smoke" and not bool(train_feat.valid[:, cfg.context:].any()):
        raise ValueError("There are no valid training futures. Check clip length and stride.")

    return train_feat

## 7. Define the two prediction heads

The masked observation GRU returns h_obs. The view MLP reads h_obs, the appearance anchor, and source/destination camera embeddings. A separate GRU starts from h_obs and autoregressively predicts three later embeddings. With angular error e and positive scale v, each predictive term is e/v + 0.5 log(v).

In [7]:
# 7. Define the recurrent world model and the paired prediction losses.
def last_valid(features, mask):
    indices = torch.arange(mask.shape[1], device=mask.device)
    last = (indices * mask).amax(dim=1)
    return features[torch.arange(len(features), device=features.device), last]

class LatentWorldModel(nn.Module):
    def __init__(self, camera_context, camera_count):
        super().__init__()
        self.camera_context = camera_context
        self.camera = nn.Embedding(camera_count, 16)
        self.observe = nn.GRUCell(cfg.embed_dim + 16, cfg.hidden_dim)
        self.transition = nn.GRUCell(cfg.embed_dim + 16, cfg.hidden_dim)
        self.future_stats = nn.Linear(cfg.hidden_dim, cfg.embed_dim + 1)
        self.view_stats = nn.Sequential(
            nn.Linear(cfg.hidden_dim + cfg.embed_dim + 32, cfg.hidden_dim),
            nn.LayerNorm(cfg.hidden_dim),
            nn.SiLU(),
            nn.Linear(cfg.hidden_dim, cfg.embed_dim + 1),
        )
        for layer in (self.future_stats, self.view_stats[-1]):
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.zeros_(layer.bias)
            nn.init.constant_(layer.bias[-1:], math.log(cfg.variance_reference))

    def cam_features(self, cameras):
        features = self.camera(cameras)
        if not self.camera_context:
            return torch.zeros_like(features)
        return features

    @staticmethod
    def get_stats(parameters, anchor):
        residual, log_var = parameters[:, :-1], parameters[:, -1]
        mean = F.normalize(anchor + 0.5 * residual, dim=-1)
        return mean, log_var.clamp(min=-5.0, max=0.70)

    def encode_prefix(self, prefix, valid, cameras):
        anchor = masked_anchor(prefix, valid)
        camera = self.cam_features(cameras)
        state = prefix.new_zeros(len(prefix), cfg.hidden_dim)
        for index in range(prefix.shape[1]):
            candidate = self.observe(torch.cat((prefix[:, index], camera), dim=1), state)
            state = torch.where(valid[:, index, None], candidate, state)
        return anchor, state

    def roll(self, prefix, valid, cameras):
        anchor, observed_state = self.encode_prefix(prefix, valid, cameras)
        state = observed_state
        camera = self.cam_features(cameras)
        previous = last_valid(prefix, valid)
        means, log_vars = [], []
        for _ in range(cfg.horizon):
            state = self.transition(torch.cat((previous, camera), dim=1), state)
            previous, log_var = self.get_stats(self.future_stats(state), anchor)
            means.append(previous)
            log_vars.append(log_var)
        return anchor, observed_state, torch.stack(means, dim=1), torch.stack(log_vars, dim=1)

    def predict_view(self, state, anchor, source, target):
        features = torch.cat(
            (state, anchor, self.cam_features(source), self.cam_features(target)), dim=1
        )
        return self.get_stats(self.view_stats(features), anchor)

def angular_error(mean, target):
    return (1.0 - (mean * target).sum(dim=-1)).clamp_min(0.0)

def predictive_loss(mean, log_var, target):
    return angular_error(mean, target) * torch.exp(-log_var) + 0.5 * log_var

def view_and_future_loss(model, features, index, use_corruption, future_weight):
    clean = features.clean[index].to(device)
    corrupt = features.occluded[index].to(device)
    valid = features.valid[index].to(device)
    cameras = features.cameras[index].to(device)
    pids = features.pids[index].to(device)
    pair = paired_index(len(clean), device)

    clean_prefix = clean[:, :cfg.context]
    prefix = torch.where(use_corruption[:, None, None], corrupt, clean_prefix)
    prefix_valid = valid[:, :cfg.context]
    anchor, state, future, future_log_var = model.roll(prefix, prefix_valid, cameras)
    targets = masked_anchor(clean_prefix, prefix_valid)
    target_cameras = cameras[pair]

    mean, log_var = model.predict_view(state, anchor, cameras, target_cameras)
    view_nll = predictive_loss(mean, log_var, targets[pair]).mean()

    # Each row's distractors come from its queried destination camera.
    same_destination = target_cameras[:, None] == cameras[None, :]
    different_pid = pids[:, None] != pids[None, :]
    paired_entry = torch.arange(len(pids), device=device)[None, :] == pair[:, None]
    valid_candidates = same_destination & (different_pid | paired_entry)
    if not (valid_candidates.sum(dim=1) >= 2).all():
        raise ValueError("Missing destination-camera negative")
    logits = (mean @ targets.T) / 0.08
    view_contrast = F.cross_entropy(logits.masked_fill(~valid_candidates, -1e4), pair)

    future_mask = valid[:, cfg.context:]
    forecast = predictive_loss(future, future_log_var, clean[:, cfg.context:])
    forecast_nll = (forecast * future_mask).sum() / future_mask.sum().clamp_min(1)
    loss = view_contrast + cfg.view_loss_weight * view_nll + future_weight * forecast_nll
    return loss, view_contrast, view_nll, forecast_nll

## 8. Fit the matched models

Total loss = destination contrastive loss + 0.50 × view loss + λ × masked future loss. Use λ = 0.75 for the full model and zero for the matched control. Reset initialization, sampling and corruption seeds before every arm.

In [8]:
# 8. Train the full model and the two architectural controls.
def fit_world_model(name, camera_context, future_weight, train_feat, camera_count, seed, out_dir):
    seed_all(seed + 2000)
    model = LatentWorldModel(camera_context=camera_context, camera_count=camera_count).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=9e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.head_epochs, eta_min=1e-4
    )
    sampler = CameraPairBatchSampler(train_feat.pids.numpy(), train_feat.cameras.numpy(), seed + 2010)
    noise_generator = torch.Generator().manual_seed(seed + 2020)
    history = []
    for epoch in range(cfg.head_epochs):
        sampler.epoch = epoch
        means = np.zeros(4, dtype=np.float64)
        for batch in sampler:
            index = torch.tensor(batch, dtype=torch.long)
            corruption = torch.rand(len(index), generator=noise_generator)
            use_corruption = (corruption < cfg.prefix_occlusion_probability).to(device)
            optimizer.zero_grad(set_to_none=True)
            losses = view_and_future_loss(model, train_feat, index, use_corruption, future_weight)
            if not bool(torch.isfinite(torch.stack(losses)).all()):
                raise FloatingPointError(f"Non-finite training loss in {name}.")
            losses[0].backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            means += torch.stack(losses).detach().cpu().numpy()
        scheduler.step()
        means /= len(sampler)
        history.append((name, epoch + 1, *means.tolist()))
        print(f"{name} epoch {epoch + 1:02d}/{cfg.head_epochs:02d} | loss {means[0]:.4f}")

    atomic_checkpoint({
        "model_state": model.state_dict(),
        "settings": asdict(cfg),
        "camera_context": camera_context,
        "camera_count": camera_count,
        "future_weight": future_weight,
    }, out_dir / f"{name}.pt")
    return model.eval().requires_grad_(False), history

## 9. Rank prefix pairs and calculate retrieval metrics

Each direction adds its predicted cosine improvement over appearance, scaled by 0.20/(0.20 + predicted scale). Average both corrections. Same-camera pairs retain appearance scores. Hit-based mAP and trapezoidal mAP are saved separately.

In [9]:
# 9. Define retrieval scores and exact query-level metrics.
@dataclass
class RetrievalState:
    anchors: torch.Tensor
    cameras: torch.Tensor
    view_means: torch.Tensor | None = None
    view_vars: torch.Tensor | None = None

@torch.no_grad()
def represent(data, model=None, condition="clean", reverse=False):
    prefix = data.clean[:, :cfg.context] if condition == "clean" else data.occluded
    mask = data.valid[:, :cfg.context]
    anchors = masked_anchor(prefix, mask)
    if model is None:
        return RetrievalState(anchors, data.cameras)
    if reverse:
        prefix, mask = prefix.flip(1), mask.flip(1)

    means, variances = [], []
    camera_count = model.camera.num_embeddings
    destination = torch.arange(model.camera.num_embeddings, device=device)
    for start in range(0, len(prefix), 384):
        take = slice(start, start + 384)
        cameras = data.cameras[take].to(device)
        _, state = model.encode_prefix(prefix[take].to(device), mask[take].to(device), cameras)
        anchor = anchors[take].to(device)
        mean, log_var = model.predict_view(
            state.repeat_interleave(camera_count, dim=0),
            anchor.repeat_interleave(camera_count, dim=0),
            cameras.repeat_interleave(camera_count),
            destination.repeat(len(anchor)),
        )
        means.append(mean.reshape(len(anchor), camera_count, -1).cpu())
        variances.append(log_var.exp().reshape(len(anchor), camera_count).cpu())
    return RetrievalState(anchors, data.cameras, torch.cat(means), torch.cat(variances))

@torch.no_grad()
def score_chunks(query, gallery, use_gate, bidirectional, query_batch=64):
    gallery_anchor = gallery.anchors.to(device)
    gallery_groups = {
        cam: torch.where(gallery.cameras == cam)[0].to(device)
        for cam in gallery.cameras.unique().tolist()
    }
    use_world = query.view_means is not None
    if use_world and bidirectional:
        gallery_means = gallery.view_means.to(device)
        gallery_vars = gallery.view_vars.to(device)
    for start in range(0, len(query.anchors), query_batch):
        take = slice(start, start + query_batch)
        query_anchor = query.anchors[take].to(device)
        base = query_anchor @ gallery_anchor.T
        correction = torch.zeros_like(base)
        if use_world:
            query_means = query.view_means[take].to(device)
            query_vars = query.view_vars[take].to(device)
            query_cameras = query.cameras[take]
            for source in query_cameras.unique().tolist():
                q_index = torch.where(query_cameras == source)[0].to(device)
                for target, g_index in gallery_groups.items():
                    if source == target:
                        continue
                    pos = (q_index[:, None], g_index[None, :])
                    forward = query_means[q_index, target] @ gallery_anchor[g_index].T - base[pos]
                    if use_gate:
                        forward *= (cfg.variance_reference / (
                            cfg.variance_reference + query_vars[q_index, target]
                        ))[:, None]
                    if not bidirectional:
                        correction[pos] = forward
                        continue
                    backward = query_anchor[q_index] @ gallery_means[g_index, source].T - base[pos]
                    if use_gate:
                        backward *= (cfg.variance_reference / (
                            cfg.variance_reference + gallery_vars[g_index, source]
                        ))[None, :]
                    correction[pos] = 0.5 * (forward + backward)
        yield start, base.cpu().numpy(), correction.cpu().numpy()

def average_precisions(relevant_ranks):
    hit_counts = np.arange(1, len(relevant_ranks) + 1, dtype=np.float64)
    precision = hit_counts / relevant_ranks
    previous = np.ones_like(precision)
    np.divide(hit_counts - 1, relevant_ranks - 1, out=previous, where=relevant_ranks > 1)
    return float(precision.mean()), float(0.5 * (precision + previous).mean())

def evaluate_rankings(query_data, gallery_data, query_repr, gallery_repr, alphas, use_gate, bidirectional):
    query_pids = query_data.pids.numpy()
    query_cameras = query_data.cameras.numpy()
    gallery_pids = gallery_data.pids.numpy()
    gallery_cameras = gallery_data.cameras.numpy()
    result = {alpha: [] for alpha in alphas}
    for start, baseline, delta in score_chunks(query_repr, gallery_repr, use_gate, bidirectional):
        for row, (base, correction) in enumerate(zip(baseline, delta)):
            q = start + row
            pid, cam = int(query_pids[q]), int(query_cameras[q])
            same_identity = gallery_pids == pid
            keep = (gallery_pids >= 0) & ~(same_identity & (gallery_cameras == cam))
            valid_positive = same_identity[keep]
            if not valid_positive.any():
                continue
            for alpha in alphas:
                scores = base[keep] + alpha * correction[keep]
                ranking = np.argsort(-scores, kind="stable")
                relevant_ranks = np.flatnonzero(valid_positive[ranking]) + 1
                ap, ap_trapezoidal = average_precisions(relevant_ranks)
                result[alpha].append((query_data.keys[q], pid, int(relevant_ranks[0]), ap, ap_trapezoidal))
    columns = ["query_key", "query_pid", "first_rank", "ap", "ap_trapezoidal"]
    records = {alpha: pd.DataFrame(rows, columns=columns) for alpha, rows in result.items()}
    if not records[alphas[0]].shape[0]:
        raise ValueError("None of the queries has a different-camera positive in its gallery.")
    return records

def summarize_retrieval(rows, total_queries):
    return {
        "mAP": rows["ap"].mean(),
        "mAP_trapezoidal": rows["ap_trapezoidal"].mean(),
        "rank1": (rows["first_rank"] <= 1).mean(),
        "rank5": (rows["first_rank"] <= 5).mean(),
        "rank10": (rows["first_rank"] <= 10).mean(),
        "valid_queries": len(rows),
        "skipped_without_positive": total_queries - len(rows),
    }

## 10. Choose a common fusion weight

Maximize average validation hit-mAP across the full and no-future models and both conditions, with the smaller weight breaking ties. Apply this same weight to the ablations.

In [10]:
# 10. Freeze scoring hyperparameters using only validation identities.
@dataclass(frozen=True)
class Experiment:
    name: str
    model: str | None
    use_gate: bool = True
    bidirectional: bool = True
    reverse: bool = False

experiments = [
    Experiment("Appearance cosine", None),
    Experiment("World model", "world_model"),
    Experiment("View-only control", "view_only"),
    Experiment("World model without variance gate", "world_model", use_gate=False),
    Experiment("World model with forward scoring", "world_model", bidirectional=False),
    Experiment("Camera-blind control", "camera_blind"),
    Experiment("World model with reversed context", "world_model", reverse=True),
]
conditions = ("clean", "occluded")
alpha_grid = (0.0, 0.25, 0.5, 0.75, 1.0, 1.5)

def select_alpha(vq,vg,models):
    records = []
    for arm in ('world_model','view_only'):
        model = models[arm]
        gallery = represent(vg,model)
        for cond in conditions:
            q = represent(vq,model,cond)
            by_alpha = evaluate_rankings(vq,vg,q,gallery,alpha_grid,True,True)
            records.extend((arm,cond,a,rows.ap.mean()) for a,rows in by_alpha.items())
    grid = pd.DataFrame(records,columns=['arm','condition','alpha','hit_mAP'])
    mean = grid.groupby('alpha',sort=True).hit_mAP.mean()
    return float(mean.idxmax()),grid

## 11. Check forecast skill

Measure masked cosine error against last-observed-frame persistence at each horizon. Future variance bins describe only the temporal head.

In [11]:
@torch.no_grad()
def diagnose_future(query_data, model):
    horizon_rows, calibration_rows = [], []
    for condition in conditions:
        prefix = query_data.clean[:, :cfg.context] if condition == "clean" else query_data.occluded
        mask = query_data.valid
        forecast_parts, persistence_parts, variance_parts = [], [], []
        for start in range(0, len(prefix), 384):
            take = slice(start, start + 384)
            context = prefix[take].to(device)
            context_mask = mask[take, :cfg.context].to(device)
            cameras = query_data.cameras[take].to(device)
            target = query_data.clean[take, cfg.context:].to(device)
            _, _, prediction, log_var = model.roll(context, context_mask, cameras)
            persistence = last_valid(context, context_mask)[:, None]
            forecast_parts.append(angular_error(prediction, target).cpu())
            persistence_parts.append(angular_error(persistence, target).cpu())
            variance_parts.append(log_var.exp().cpu())
        forecast = torch.cat(forecast_parts)
        persistence = torch.cat(persistence_parts)
        variance = torch.cat(variance_parts)
        future_mask = mask[:, cfg.context:]

        for step in range(cfg.horizon):
            valid = future_mask[:, step]
            if not bool(valid.any()):
                continue
            for method, errors in (("World model", forecast), ("Last-frame persistence", persistence)):
                horizon_rows.append((
                    condition, method, step + 1, int(valid.sum()), float(errors[valid, step].mean())
                ))

        # Gaussian-form loss implies mean variance near twice the mean angular error.
        errors = (2.0 * forecast[future_mask]).numpy()
        var = variance[future_mask].numpy()
        if len(var):
            for i, group in enumerate(np.array_split(np.argsort(var), min(5, len(var))), 1):
                calibration_rows.append((condition, i, len(group), var[group].mean(), errors[group].mean()))
    return (
        pd.DataFrame(horizon_rows, columns=[
            "query_condition", "method", "horizon", "valid_targets", "mean_cosine_error"
        ]),
        pd.DataFrame(calibration_rows, columns=[
            "query_condition", "variance_bin", "valid_targets", "mean_variance", "twice_mean_error"
        ]),
    )

## 12. Run the dataset experiment

This cell trains the encoder and three heads, tunes on validation identities, then evaluates each method on both query conditions. Every selected split and seed gets its own checkpoints and CSVs.

In [12]:
def run_experiment(data, split, seed):
    train,vq_tracks,vg_tracks,tq_tracks,tg_tracks,paths,ncam,protocol = data
    out = OUTPUT / f"split_{split:02d}" / f"seed_{seed}"
    out.mkdir(parents=True,exist_ok=True)
    seed_all(seed)
    manifest = dict(dataset=DATASET, profile=RUN_PROFILE, seed=seed, split=split,
                    split_seed=SPLIT_SEED, protocol=protocol, settings=asdict(cfg),
                    torch_version=str(torch.__version__),device=str(device),cameras=ncam)
    splits = {'train':train,'validation_query':vq_tracks,'validation_gallery':vg_tracks,
              'test_query':tq_tracks,'test_gallery':tg_tracks}
    pd.DataFrame([(part,t.key,t.pid,t.cam,len(t.frames)) for part,tracks in splits.items() for t in tracks],
                 columns=['partition','track_key','pid','camera','length']).to_csv(out/'partitions.csv',index=False)
    encoder = train_encoder(train,paths,seed,out)
    features = cache_train(encoder,paths,train)
    models,history = {},[]
    for name,cam,weight in [('world_model',True,cfg.forecast_loss_weight),('view_only',True,0.0),
                            ('camera_blind',False,cfg.forecast_loss_weight)]:
        models[name],curve = fit_world_model(name,cam,weight,features,ncam,seed,out)
        history.extend(curve)
    pd.DataFrame(history,columns=['arm','epoch','loss','contrast','view','future']).to_csv(out/'head_losses.csv',index=False)
    del features
    vq,vg = cache_eval(encoder,paths,vq_tracks,vg_tracks,'Validation')
    alpha,grid = select_alpha(vq,vg,models)
    grid.to_csv(out/'validation_grid.csv',index=False)
    del vq,vg
    manifest['shared_alpha'] = alpha
    tq,tg = cache_eval(encoder,paths,tq_tracks,tg_tracks,'Test')
    rows,summary = [],[]
    for trial in experiments:
        model = models.get(trial.model)
        gallery = represent(tg,model,reverse=trial.reverse)
        for cond in conditions:
            query = represent(tq,model,cond,reverse=trial.reverse)
            weight = 0.0 if model is None else alpha
            df = evaluate_rankings(tq,tg,query,gallery,(weight,),trial.use_gate,trial.bidirectional)[weight]
            rows.append(df.assign(method=trial.name,condition=cond,split=split,seed=seed))
            summary.append(dict(method=trial.name,condition=cond,split=split,seed=seed,
                                **summarize_retrieval(df,len(tq.keys))))
    # Verify that retrieval depends only on the prefix by changing the suffix.
    sample = tq.take(range(min(4,len(tq.keys))))
    changed = sample.take(range(len(sample.keys)))
    changed.clean[:,cfg.context:] = 0
    first = represent(sample,models['world_model'])
    second = represent(changed,models['world_model'])
    torch.testing.assert_close(first.view_means,second.view_means)
    torch.testing.assert_close(first.view_vars,second.view_vars)
    # Assess the actual retrieval gate on held-out identity-matched view pairs.
    # Average all positives within each destination before binning query-camera pairs.
    view_rows = []
    groups = defaultdict(list)
    for i,(pid,cam) in enumerate(zip(tg.pids.tolist(),tg.cameras.tolist())):
        if pid>0:
            groups[(pid,cam)].append(i)
    anchors = represent(tg).anchors
    for cond in conditions:
        state = represent(tq,models['world_model'],cond)
        for i,(pid,cam) in enumerate(zip(tq.pids.tolist(),tq.cameras.tolist())):
            for dest in range(ncam):
                pos = groups.get((pid,dest))
                if dest==cam or not pos:
                    continue
                err = 2*(1-anchors[pos] @ state.view_means[i,dest]).clamp_min(0).mean()
                view_rows.append((cond,tq.keys[i],cam,dest,float(state.view_vars[i,dest]),float(err)))
    pd.DataFrame(view_rows, columns=['condition','query_key','source_camera','target_camera',
                 'predicted_scale','twice_mean_angular_error']).to_csv(out/'view_calibration.csv',index=False)
    details,bins = diagnose_future(tq,models['world_model'])
    details.to_csv(out/'future_vs_persistence.csv',index=False)
    bins.to_csv(out/'temporal_scale_bins.csv',index=False)
    metric = pd.DataFrame(summary).rename(columns={'mAP':'mAP_hit'})
    per_query = pd.concat(rows,ignore_index=True)
    metric.to_csv(out/'retrieval.csv',index=False)
    per_query.to_csv(out/'queries.csv',index=False)
    (out/'run.json').write_text(json.dumps(manifest,indent=2))
    return metric,per_query

run_metrics, run_queries = [], []
for split in SPLITS:
    partitions = load_split(split)
    for seed in SEEDS:
        print(f"Running {DATASET}, {RUN_PROFILE}, split {split}, seed {seed}")
        metrics,queries = run_experiment(partitions,split,seed)
        run_metrics.append(metrics)
        run_queries.append(queries)
        display(metrics.round(4))

artificial pipeline check | train / validation / test identities: 14 2 8
Running MARS, smoke, split 0, seed 42
Encoder epoch 01/02 | loss 5.3238
Encoder epoch 02/02 | loss 3.4403
Train windows A: 84 clips | with future frames 100.0% | 0.4s
Train windows B: 84 clips | with future frames 100.0% | 0.2s
world_model epoch 01/03 | loss 0.2435
world_model epoch 02/03 | loss 0.2613
world_model epoch 03/03 | loss 0.2194
view_only epoch 01/03 | loss 0.7409
view_only epoch 02/03 | loss 0.7415
view_only epoch 03/03 | loss 0.7199
camera_blind epoch 01/03 | loss 0.2367
camera_blind epoch 02/03 | loss 0.2401
camera_blind epoch 03/03 | loss 0.1850
Validation gallery: 12 clips | with future frames 100.0% | 0.1s
Test gallery: 48 clips | with future frames 100.0% | 0.1s


,method,condition,split,seed,mAP_hit,mAP_trapezoidal,rank1,rank5,rank10,valid_queries,skipped_without_positive
0,Appearance cosine,clean,0,42,0.8513,0.8422,1.0,1.000,1.000,8,0
1,Appearance cosine,occluded,0,42,0.4890,0.4596,0.5,0.625,0.875,8,0
2,World model,clean,0,42,0.8513,0.8422,1.0,1.000,1.000,8,0
3,World model,occluded,0,42,0.4890,0.4596,0.5,0.625,0.875,8,0
4,View-only control,clean,0,42,0.8513,0.8422,1.0,1.000,1.000,8,0
5,View-only control,occluded,0,42,0.4890,0.4596,0.5,0.625,0.875,8,0
6,World model without variance gate,clean,0,42,0.8513,0.8422,1.0,1.000,1.000,8,0
7,World model without variance gate,occluded,0,42,0.4890,0.4596,0.5,0.625,0.875,8,0
8,World model with forward scoring,clean,0,42,0.8513,0.8422,1.0,1.000,1.000,8,0
9,World model with forward scoring,occluded,0,42,0.4890,0.4596,0.5,0.625,0.875,8,0


## 13. Summarize performance and paired uncertainty

Metrics are percentages. Average splits within each seed before calculating the across-seed mean and sample SD. One seed gives an undefined SD. Paired intervals resample identities, retaining their appearances across splits and seeds, conditional on the trained models and fixed galleries.

In [13]:
metrics = pd.concat(run_metrics,ignore_index=True)
query_results = pd.concat(run_queries,ignore_index=True)
measures = ['mAP_hit','rank1','rank5','rank10','mAP_trapezoidal']
per_seed = metrics.groupby(['method','condition','seed'])[measures].mean()*100
summary = per_seed.groupby(['method','condition']).agg(['mean','std'])
summary.to_csv(OUTPUT/'mean_sd_percent.csv')
display(summary.round(2))

def primary_intervals(records, draws=1500):
    results = []
    for condition,sub in records.groupby('condition'):
        sub = sub.assign(rank1_hit=(sub.first_rank<=1).astype(float))
        keys = ['split','seed','query_pid','query_key']
        for metric in ['ap','rank1_hit']:
            wide = sub.pivot(index=keys,columns='method',values=metric)
            paired = wide[['World model','View-only control']]
            if paired.isna().any().any():
                raise ValueError("The matched models must use identical queries")
            delta = (paired['World model']-paired['View-only control']).rename('gap').reset_index()
            # Identical identities can recur across the official iLIDS splits.
            by_id = delta.groupby(['query_pid','split']).gap.agg(['sum','count']).reset_index()
            sums = by_id.pivot(index='query_pid',columns='split',values='sum').fillna(0).to_numpy()
            counts = by_id.pivot(index='query_pid',columns='split',values='count').fillna(0).to_numpy()
            rng = np.random.default_rng(1729)
            boot = []
            for _ in range(draws):
                sample = rng.integers(len(sums),size=len(sums))
                denom = counts[sample].sum(axis=0)
                if (denom>0).all():
                    boot.append((sums[sample].sum(axis=0)/denom).mean())
            point = delta.groupby(['split','seed']).gap.mean().mean()
            low,high = np.quantile(boot,[0.025,0.975])
            results.append((condition,metric,100*point,100*low,100*high,len(sums)))
    return pd.DataFrame(results,columns=['condition','endpoint','delta_pp','low95_pp','high95_pp','identities'])

paired = primary_intervals(query_results)
paired.to_csv(OUTPUT/'full_minus_no_future.csv',index=False)
display(paired.round(2))
print("Output directory:",OUTPUT)

mAP_hit      rank1      rank5      \
                                               mean std   mean std   mean std   
method                            condition                                     
Appearance cosine                 clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   
Camera-blind control              clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   
View-only control                 clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   
World model                       clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   
World model with forward scoring  clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   
World model with reversed context clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   
World model without variance gate clean       85.13 NaN  100.0 NaN  100.0 NaN   
                                  occluded    48.90 NaN   50.0 NaN   62.5 NaN   

                                            rank10     mAP_trapezoidal      
                                              mean std            mean std  
method                            condition                                 
Appearance cosine                 clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN  
Camera-blind control              clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN  
View-only control                 clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN  
World model                       clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN  
World model with forward scoring  clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN  
World model with reversed context clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN  
World model without variance gate clean      100.0 NaN           84.22 NaN  
                                  occluded    87.5 NaN           45.96 NaN

,condition,endpoint,delta_pp,low95_pp,high95_pp,identities
0,clean,ap,0.0,0.0,0.0,8
1,clean,rank1_hit,0.0,0.0,0.0,8
2,occluded,ap,0.0,0.0,0.0,8
3,occluded,rank1_hit,0.0,0.0,0.0,8


Output directory: /kaggle/working/reid/MARS/smoke


## Interpreting the run

Read the full-minus-no-future differences first. A positive difference favors future supervision. A confidence interval spanning zero leaves the direction uncertain. Compare forecast error with persistence before interpreting temporal skill. An alpha of zero gives the appearance ranking for all methods. The synthetic rectangle is a reproducible stress test; “clean” means the unmodified crops. Use research-profile results and report the number of evaluated queries, seeds and splits.